# 핸즈온 00 — 환경 설정 & 첫 호출

**소요 시간**: 15~20분
**학습 목표**:
1. Colab에서 Gemini API를 안전하게 호출하기 위한 환경을 만든다
2. AI Studio에서 API 키를 발급받고 Colab Secret으로 등록한다
3. Gemini 3 시리즈 모델을 호출해 응답을 받아본다
4. 토큰 사용량과 latency를 측정하는 패턴을 익힌다

> **Note**: 이 핸즈온 시리즈는 유티정보 ITS 8시간 특강용으로 설계되었습니다. 모든 데이터는 공개 데이터셋이며, API 키는 각자 발급받아 사용합니다.

## 0-1. API 키 발급 (사전 준비)

> 강의 시작 전 미리 발급해두셨다면 0-2로 건너뛰세요.

### 발급 절차
1. [https://aistudio.google.com](https://aistudio.google.com) 접속 (Google 계정 필요)
2. 좌측 상단 **"Get API key"** 클릭 → **"Create API key"**
3. 발급된 키 문자열(`AIza...` 으로 시작)을 안전한 곳에 복사

### 무료 티어 한도 (2026년 5월 기준)
| 모델 | RPM | RPD |
|---|---|---|
| Gemini 3.1 Pro Preview | (AI Studio Web만 무료, API 무료티어 없음) | - |
| Gemini 3 Flash Preview | 10 | 250 |
| Gemini 3.1 Flash-Lite Preview | 15 | 1000 |

> 8시간 강의 동안 **Flash-Lite 위주**로 실습합니다. Pro는 비교 데모 용도로만 사용합니다.

### 보안 주의
- API 키를 노트북 코드에 직접 붙여넣지 마세요 (GitHub에 올라가면 자동 폐기됨)
- 본 노트북은 **Colab Secret** 기능을 사용해 안전하게 관리합니다

## 0-2. Colab Secret에 API 키 등록

1. Colab 좌측 사이드바의 **🔑 (열쇠) 아이콘** 클릭
2. **"새 보안 비밀 추가"** 클릭
3. 다음과 같이 입력:
   - 이름: `GEMINI_API_KEY`
   - 값: 발급받은 API 키
4. **"노트북 액세스"** 토글을 **켭니다** (이걸 안 켜면 노트북에서 못 읽음)

등록 후 아래 셀을 실행하면 키를 안전하게 불러올 수 있습니다.

In [ ]:
# Colab Secret에서 API 키 불러오기
import os

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    print(f"✅ API 키 로드 성공 (마스킹: {GEMINI_API_KEY[:8]}...{GEMINI_API_KEY[-4:]})")
except ImportError:
    # 로컬 Jupyter 환경 fallback
    if "GEMINI_API_KEY" in os.environ:
        print("✅ 환경변수에서 API 키 로드")
    else:
        print("❌ Colab이 아니거나 키가 등록되지 않았습니다")
        print("   직접 입력하려면: os.environ['GEMINI_API_KEY'] = 'your-key'")

## 0-3. Gemini SDK 설치

`google-genai`가 새 통합 SDK입니다. (이전 `google-generativeai`는 deprecated)

In [ ]:
!pip install -q -U google-genai

## 0-4. 첫 호출 — Hello, Gemini

가장 가벼운 모델인 **Gemini 3.1 Flash-Lite**로 시작합니다.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents="ITS(지능형교통체계)에서 노드와 링크가 무엇인지 한 문장씩 설명해줘.",
)

print(response.text)

### 응답 메타데이터 확인

토큰 사용량과 모델 버전을 항상 확인하는 습관을 들이세요. 비용·성능 디버깅의 출발점입니다.

In [ ]:
def show_usage(response, label=""):
    """토큰 사용량을 보기 좋게 출력."""
    u = response.usage_metadata
    print(f"--- {label} ---")
    print(f"  Input  tokens: {u.prompt_token_count:>6,}")
    print(f"  Output tokens: {u.candidates_token_count:>6,}")
    if hasattr(u, "thoughts_token_count") and u.thoughts_token_count:
        print(f"  Thinking     : {u.thoughts_token_count:>6,}  ⚠️ 비용 청구됨")
    print(f"  Total        : {u.total_token_count:>6,}")
    print()

show_usage(response, "Flash-Lite, default thinking")

## 0-5. Gemini 3 시리즈 모델 라인업 확인

자료에서 다룬 모델 3종을 모두 호출해보고 응답 차이를 확인합니다.

In [ ]:
import time

MODELS = [
    "gemini-3.1-pro-preview",
    "gemini-3-flash-preview",
    "gemini-3.1-flash-lite-preview",
]

PROMPT = "도로 정체의 원인을 3가지 카테고리로 한 문장씩 정리해줘."

for model in MODELS:
    t0 = time.time()
    try:
        resp = client.models.generate_content(model=model, contents=PROMPT)
        latency = time.time() - t0
        print(f"\n=== {model} ===")
        print(f"⏱  {latency:.2f}s")
        show_usage(resp, model)
        print(resp.text[:200], "...")
    except Exception as e:
        print(f"❌ {model}: {e}\n")


> **관찰 포인트**
>
> - Pro는 무료 API 티어가 없어 키에 결제수단이 등록되지 않았다면 401/403이 날 수 있습니다 → 실패하면 실습은 Flash / Flash-Lite 두 개로만 진행
> - 같은 프롬프트인데 Pro의 응답이 가장 길고 구조적, Flash-Lite는 짧고 직설적입니다
> - **이 차이가 비용·품질 트레이드오프의 출발점**입니다 — 다음 핸즈온에서 정량화합니다

## 0-6. Thinking Level 파라미터 시연

Gemini 3의 가장 큰 변화 중 하나입니다. **명시하지 않으면 기본값이 `high`**라서 비용이 의도치 않게 올라갑니다.

In [ ]:
def call_with_thinking(model, level, prompt):
    """thinking_level을 지정해 호출."""
    t0 = time.time()
    config = types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level=level)
    )
    resp = client.models.generate_content(
        model=model, contents=prompt, config=config
    )
    latency = time.time() - t0
    return resp, latency


COMPLEX_PROMPT = (
    "교차로 신호 시간을 1초 단축하면 평균 통행시간이 어떻게 변하는지, "
    "도로공학적 관점에서 단계적으로 추론해줘."
)

for level in ["low", "medium", "high"]:
    resp, latency = call_with_thinking(
        "gemini-3-flash-preview", level, COMPLEX_PROMPT
    )
    print(f"\n=== thinking_level = {level} | {latency:.2f}s ===")
    show_usage(resp, level)


> **체크 포인트**
>
> 1. `low` → `medium` → `high`로 갈수록 thinking 토큰이 급증합니다
> 2. thinking 토큰은 출력 토큰과 동일 가격으로 청구됩니다 (3.1 Pro 기준 $12/1M)
> 3. 단순 작업에 `high`를 쓰는 건 곧 비용 누수입니다
>
> 다음 핸즈온에서 모델 × thinking level 9개 조합을 모두 측정해 매트릭스를 만듭니다.

## 0-7. 트러블슈팅 체크리스트

문제가 발생하면 이 순서로 확인하세요.

| 증상 | 원인 | 해결 |
|---|---|---|
| `404 Not Found` | 모델명 오타 또는 셧다운된 모델 | 모델명 정확히 확인. `gemini-1.5-*`, `gemini-2.0-*`은 셧다운됨 |
| `403 PERMISSION_DENIED` | API 키 미등록 또는 결제수단 필요 (Pro) | AI Studio에서 키 재발급 / Pro는 결제수단 등록 |
| `429 RESOURCE_EXHAUSTED` | 분당/일일 호출 한도 초과 | 1분 대기 또는 더 가벼운 모델로 변경 |
| 응답 텍스트가 비어있음 | safety 필터 또는 빈 응답 | `response.candidates[0].finish_reason` 확인 |
| Colab Secret 못 읽음 | "노트북 액세스" 토글 OFF | Colab 좌측 🔑에서 토글 ON |

## 0-8. 정리

- ✅ Colab + AI Studio 환경 셋업 완료
- ✅ Gemini 3 시리즈 3개 모델 호출 성공
- ✅ thinking_level 파라미터 동작 확인
- ✅ 토큰 사용량 측정 패턴 확보

다음 핸즈온(`01_model_matrix.ipynb`)에서는 **9개 조합으로 비용·성능 매트릭스**를 만들어 ITS 워크로드별 모델 선택 기준을 잡습니다.